In [ ]:
import sys
import datetime

sys.path.append("..")

## Connect DB

In [ ]:
from src.setup.setup import setup
from src.config import DATABASE_PATH

setup(DATABASE_PATH.absolute().as_posix())

# Get Data

In [ ]:
import ezyquant as ez
from ezyquant.reader import _SETDataReaderCached
import ezyquant.fields as fld

In [ ]:
sdr = _SETDataReaderCached()
start_date_data = datetime.date(2019, 1, 1)  # ISO format: 2019-01-01
start_date_backtest = datetime.date(2020, 1, 1)  # ISO format: 2020-01-01
end_date = datetime.date.fromisoformat(sdr.last_update())

In [ ]:
ssc = ez.SETSignalCreator(
    start_date=start_date_data.isoformat(),
    end_date=end_date.isoformat(),
    index_list=["BANK"],
)

In [ ]:
close_df = ssc.get_data(field="close", timeframe="daily")
high_df = close_df.copy(deep=True)
low_df = close_df.copy(deep=True)
close_df

## Strategy

In [ ]:
ema_150_df = ssc.ta.ema(close_df, 150)
ema_200_df = ssc.ta.ema(close_df, 200)

(
    dc_252_high_df,
    dc_252_low_df,
    dc_252_middle_df,
    dc_252_percentage_df,
    dc_252_band_width_df,
) = ssc.ta.dc(high_df, low_df, close_df, 252)

In [ ]:
from src.strategies.trend_template import trend_template

In [ ]:
# signal_df = trend_template(ssc, close_df, high_df, low_df)

In [ ]:
pe = ssc.get_data(field="pe", timeframe="daily")
pe.head(5)

In [ ]:
median_market_pe = pe.transpose().median()
median_market_pe

In [ ]:
median_market_pe.describe()

In [ ]:
median_market_pe.plot()  # Mean PE spikes in late 2020 due to COVID, so we use median instead

In [ ]:
median_market_pe.rolling(28).mean().plot()

In [ ]:
for date, is_pass_threshold in zip(pe.columns, (pe < 14.7).loc["2023-12-28"]):
    print(f"{date}: {is_pass_threshold}")

In [ ]:
pe.lt(median_market_pe * 0.8, axis=0)

In [ ]:
POS_NUM = 20  # TODO: Remove magic number
signal_bank_df = ssc.rank(factor_df=pe, quantity=POS_NUM, ascending=True)
signal_bank_df

In [ ]:
# TODO: Select 20 companies max for manageable diverse portfolio

In [ ]:
from src.strategies.valuation_factor import valuation_strategy

In [ ]:
pb, pe, ev_div_ebitda, fcf_yield = valuation_strategy(ssc)

In [ ]:
pb

## Backtest

In [ ]:
from src.backtest import get_backtest_algorithm

In [ ]:
INITIAL_CASH = 50000
PCT_COMMISSION = 0.15
PCT_SELL_SLIP = 0.01

In [ ]:
result = ez.backtest(
    signal_df=signal_df,
    backtest_algorithm=get_backtest_algorithm(pos_num=20),  # TODO: Remove magic number
    start_date=start_date_backtest.isoformat(),
    end_date=end_date.isoformat(),
    initial_cash=INITIAL_CASH,
    pct_commission=PCT_COMMISSION,
    pct_sell_slip=PCT_SELL_SLIP,
    price_match_mode="weighted",
    signal_delay_bar=1
)

In [ ]:
set_index_df = sdr.get_data_index_daily(
    field=fld.D_INDEX_CLOSE,
    index_list=[fld.MARKET_SET],
    start_date=start_date_data.isoformat(),
    end_date=end_date.isoformat(),
)

set_index_return = set_index_df.pct_change().fillna(0.0)

In [ ]:
result.to_basic(benchmark=set_index_return)